# Early Injury-Risk Signal Pipeline
## EP-TSP — Elbow/Shoulder IL Cohort, 2025-2026

**Question:** in the weeks before a real, documented elbow/shoulder IL placement, do Statcast-derived signals (velocity decline, spin decline, release-point variability) look statistically different from an equivalent window in healthy comparison pitchers?

**Design:** publicly-sourced cohort of injury cases (MLB.com Transactions, press reports, dated and sourced individually) vs. a control cohort of pitchers with full healthy seasons. Pre-event window: 42 days before the IL date (or reference date for controls). Signals: linear-regression slopes of velocity/spin/extension over the window, plus release-point standard deviation.

**Guardrail:** this is exploratory / hypothesis-generating analysis on a small, explicitly-sized cohort — NOT a diagnostic tool, NOT a production model, and NOT a claim of causality. Bat/pitch-tracking proxies describe external performance behavior, not internal biomechanics or medical status. Any real application would require a team's medical and performance staff, a much larger cohort, and prospective (not retrospective) validation.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
from config import INJURY_COHORT, CONTROL_COHORT, PRE_EVENT_WINDOW_DAYS
from data_loader import (
    build_injury_cohort_dataset, build_control_cohort_dataset, save_raw, load_raw
)
from features import build_signal_table
from analysis import build_comparison_report
from visualization import plot_velocity_trajectories, plot_signal_boxplot

print(f'Injury cases in cohort: {len(INJURY_COHORT)}')
print(f'Control cases in cohort: {len(CONTROL_COHORT)}')
print(f'Pre-event window: {PRE_EVENT_WINDOW_DAYS} days')

## Step 1 — Acquisition (resolves player names to MLBAM IDs automatically)

In [ ]:
# Uncomment to download live (requires internet; several minutes due to per-player API calls)
# injury_df = build_injury_cohort_dataset()
# save_raw(injury_df, 'injury_cohort_raw.csv')
# control_df = build_control_cohort_dataset()
# save_raw(control_df, 'control_cohort_raw.csv')

injury_df = load_raw('injury_cohort_raw.csv')
control_df = load_raw('control_cohort_raw.csv')
df = pd.concat([injury_df, control_df], ignore_index=True)
df.head()

## Step 2 — Per-Case Signals (velocity/spin slope, release variability)

In [ ]:
signal_df = build_signal_table(df)
signal_df

`sufficient_data` flags cases with too few pitches or too short an active window (common for early-season injuries, where there isn't much in-season data before the event) — these are excluded from the statistical comparison below, not silently averaged in.

## Step 3 — Velocity Trajectories (visual check)

In [ ]:
plot_velocity_trajectories(df)

## Step 4 — Statistical Comparison: Injury vs Control

In [ ]:
comparison = build_comparison_report(signal_df)
comparison

Mann-Whitney U (non-parametric, appropriate for small/non-normal samples) + Cohen's d effect size. With n this small, treat any p<0.05 as suggestive — worth a follow-up with a larger cohort — not as confirmatory evidence.

In [ ]:
plot_signal_boxplot(signal_df, 'velocity_slope_mph_per_day', 'velocity_slope_boxplot.png')

## Conclusion & Next Steps

This pipeline tests — on a small, transparently-sourced cohort — whether Statcast-visible signals precede real elbow/shoulder IL placements. It is a **proof-of-concept for a workflow**, not a finished predictive product. To move toward something a team could actually use operationally:

1. Expand the injury cohort to 30-50+ cases (requires systematic scraping of MLB.com Transactions + injury-type tagging)
2. Expand and properly randomize the control cohort (currently illustrative, n=2)
3. Add confounders: workload (pitch counts, days rest), role (starter/reliever), age, prior injury history
4. Validate prospectively (flag a signal in-season, track whether it precedes a real IL event) rather than only retrospectively
5. Involve a team's medical/performance staff before any signal is used to inform real decisions